# SautiCivic Bridge — OpenAI Whisper large-v3 Benchmark (Tier B Public Corpus)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spyder0000/Sauticivic/blob/main/bench/whisper_tier_b_colab.ipynb)

This notebook runs **OpenAI Whisper large-v3** on Google Colab GPU to transcribe the **60 Tier B public benchmark clips** across 3 language groups:
- **AfriSwitch** (30 clips: Pidgin-English, Yoruba-English, Hausa-English)
- **FLEURS** (15 clips: Hausa, Yoruba)
- **AfriSpeech-200** (15 clips: Nigerian-accented English)

### Colab Setup (do this first):
1. Go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (free) or A100/V100.
3. Click **Runtime > Run all** or run cells top-to-bottom.

## Step 1 — Verify GPU

In [1]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type and select a GPU."
print(f"\n✓ GPU ready: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")

Tue Sep  8 12:20:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Clone Repository

In [ ]:
# ── CONFIGURATION (edit this cell before running) ───────────────────────────
# Running from an IDE connected to Colab runtime?
# Paste your GitHub PAT here. Never commit this value — clear it before git push.
#
# Generate a PAT at: https://github.com/settings/tokens
# Required permission: Contents → Read  (on the Sauticivic repo)
#
import os
os.environ["GITHUB_TOKEN"] = "ghp_"  # <-- paste ghp_... here
#
# Alternatively, if you set the secret via the Colab web UI (colab.research.google.com)
# leave the line above empty — it will be picked up automatically from Colab Secrets.
# ────────────────────────────────────────────────────────────────────────────

In [5]:
import os
from pathlib import Path

# Resolve token: env var set above > Colab Secrets > fail loudly
GH_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()

if not GH_TOKEN:
    try:
        from google.colab import userdata
        GH_TOKEN = (userdata.get("GITHUB_TOKEN") or "").strip()
        if GH_TOKEN:
            print("✓ GitHub token loaded from Colab Secrets.")
    except Exception:
        pass

assert GH_TOKEN, (
    "No GitHub token found.\n"
    "→ IDE users: paste your PAT into the config cell above (os.environ['GITHUB_TOKEN'] = 'ghp_...')\n"
    "→ Colab browser users: add secret 'GITHUB_TOKEN' via the 🔑 icon in the left sidebar."
)

REPO_DIR = Path("/content/Sauticivic")
CLONE_URL = f"https://{GH_TOKEN}@github.com/Spyder0000/Sauticivic.git"

if not REPO_DIR.exists():
    print("Cloning Sauticivic (private repo)...")
    !git clone --quiet {CLONE_URL} /content/Sauticivic
else:
    print("Repo already present — pulling latest...")
    !git -C /content/Sauticivic pull --quiet origin main

assert REPO_DIR.exists(), "Clone failed — check your token has 'Contents: Read' permission on the repo."

%cd /content/Sauticivic
print(f"\n✓ Working directory: {os.getcwd()}")

Cloning Sauticivic (private repo)...
/content/Sauticivic

✓ Working directory: /content/Sauticivic


## Step 3 — Install Dependencies

In [6]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q openai-whisper soundfile datasets huggingface_hub python-dotenv
print("\n✓ Dependencies installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 43.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✓ Dependencies installed.


## Step 4 — Ingest Tier B Public Audio

Downloads all 60 clips from HuggingFace (~2 min on T4).  
Saved to `bench/corpus/tier_b_public/{afriswitch,fleurs,afrispeech}/` as WAV files.

> **HF token:** Only needed if you get a 401 error on gated datasets.

In [9]:
from huggingface_hub import login

In [ ]:
import os
# Uncomment and paste your HuggingFace token if you get a 401:
os.environ["HF_TOKEN"] = "hf_z"

if os.environ.get("HF_TOKEN"):
    print("✓ HF_TOKEN is set.")
else:
    print("HF_TOKEN not set — using anonymous access (fine for public datasets).")

login(os.environ["HF_TOKEN"])

✓ HF_TOKEN is set.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:
!git -C /content/Sauticivic pull origin main

From https://github.com/Spyder0000/Sauticivic
 * branch            main       -> FETCH_HEAD
Already up to date.


In [13]:
!python3 /content/Sauticivic/bench/corpus/tier_b_public/ingest_tier_b.py --dataset all

  SAUTICIVIC BRIDGE — TIER B PUBLIC DATASET INGESTION
  Mode: LIVE DOWNLOAD
  Target Dataset(s): all
  Duration Bounds: [3.0s, 60.0s]
  HF Hub Auth:       AUTHENTICATED (token prefix: hf_zRfJ...)
  Base Output Dir:   /content/Sauticivic/bench/corpus/tier_b_public

[Dataset 1: AfriSwitch] target=30 clips (10 per pair across 3 pairs)
  Source:       intronhealth/AfriSwitch (HuggingFace, CC BY-NC-SA 4.0)
  Selection:    CMI (Code-Mixing Index) descending — prioritizes high code-switching density
  Duration:     3.0s min to 60.0s max (trimmed if > 60.0s)
  Output Base:  /content/Sauticivic/bench/corpus/tier_b_public/afriswitch
  [AfriSwitch Pidgin-English] Already has 10 clips in /content/Sauticivic/bench/corpus/tier_b_public/afriswitch/pidgin/, skipping download.
  [AfriSwitch Yoruba-English] Already has 10 clips in /content/Sauticivic/bench/corpus/tier_b_public/afriswitch/yoruba/, skipping download.
  Streaming AfriSwitch [hausa] (lightweight stream, max 10 clips)...
    Saved afriswitch

In [14]:
from pathlib import Path

CORPUS_DIR = Path("bench/corpus/tier_b_public")
audio_files = sorted(CORPUS_DIR.rglob("*.wav"))

print(f"Total Tier B audio clips found: {len(audio_files)} / 60")
for subdir in ["afriswitch", "fleurs", "afrispeech"]:
    count = len(list((CORPUS_DIR / subdir).rglob("*.wav")))
    print(f"  {subdir:20s}: {count} clips")

if len(audio_files) < 60:
    print("\n⚠  Some clips missing — re-run the ingest cell or set HF_TOKEN above.")
else:
    print("\n✓ All 60 clips ready.")

Total Tier B audio clips found: 60 / 60
  afriswitch          : 30 clips
  fleurs              : 15 clips
  afrispeech          : 15 clips

✓ All 60 clips ready.


## Step 5 — Run Whisper large-v3 Transcription

Transcribes all 60 clips using GPU-accelerated Whisper `large-v3`.  
Output: `bench/results/transcripts/tier_b/whisper/<clip_id>.json`  
Expected runtime: **~10 min on T4**, ~3 min on A100.

In [17]:
# CWD is /content/Sauticivic — all paths relative to repo root
!python3 /content/Sauticivic/bench/models/run_whisper.py \
    --corpus /content/Sauticivic/bench/corpus/tier_b_public \
    --output-dir /content/Sauticivic/bench/results/transcripts \
    --model large-v3

Model:       Whisper large-v3
Corpus dir:  /content/Sauticivic/bench/corpus/tier_b_public
Output dir:  /content/Sauticivic/bench/results/transcripts/whisper
Clips found: 60

Loading Whisper large-v3 (this may take a minute on first run)...
100%|██████████████████████████████████████| 2.88G/2.88G [00:23<00:00, 131MiB/s]
Model loaded.

[1/60] afrispeech_ng_001.wav ... OK  (lang=en)  "In particular, they looked for signs of abrupt systemizing w..."
[2/60] afrispeech_ng_002.wav ... OK  (lang=en)  "Nielsen is a keen supporter of Arsenal Football Club full st..."
[3/60] afrispeech_ng_003.wav ... OK  (lang=en)  "Her 22-year-old sister also shared lives with her in Kano an..."
[4/60] afrispeech_ng_004.wav ... OK  (lang=yo)  "When Harrison Murphy sold her Tacoma condo this summer, she ..."
[5/60] afrispeech_ng_005.wav ... OK  (lang=en)  "in its brother strokes comma 40 acres and the more was the i..."
[6/60] afrispeech_ng_006.wav ... OK  (lang=en)  "For privacy reasons, call on to sign in or re

## Step 6 — Inspect Results

In [18]:
import json
from pathlib import Path

OUT_DIR = Path("bench/results/transcripts/tier_b/whisper")
transcripts = sorted(OUT_DIR.glob("*.json"))

print(f"Whisper transcripts generated: {len(transcripts)} / 60\n")

for p in transcripts[:5]:
    data    = json.loads(p.read_text(encoding="utf-8"))
    lang    = data.get("language", "?")
    err     = data.get("error")
    status  = f"ERR: {err}" if err else f"lang={lang}"
    snippet = data.get("transcript", "")[:80].replace("\n", " ")
    print(f"  [{data.get('clip_id', p.stem)}] ({status}): {snippet}...")

errors = [p.stem for p in transcripts if json.loads(p.read_text()).get("error")]
if errors:
    print(f"\nClips with errors: {errors}")

Whisper transcripts generated: 0 / 60



## Step 7 — Download Transcripts

Packages all JSON results and downloads them.  
Extract into your local repo at `bench/results/transcripts/tier_b/whisper/`.

In [20]:
!mkdir -p bench/results/transcripts/tier_b/
!mv bench/results/transcripts/whisper bench/results/transcripts/tier_b/whisper

In [27]:
!rm /content/Sauticivic/bench/results/transcripts/tier_b/whisper/synth_*.json
!ls /content/Sauticivic/bench/results/transcripts/tier_b/whisper/ | wc -l

60


In [28]:
import os
GH_TOKEN = os.environ.get("GITHUB_TOKEN", "")
!git -C /content/Sauticivic config user.email "agorotimilehin05@gmail.com"
!git -C /content/Sauticivic config user.name "Spyder0000"
!git -C /content/Sauticivic add bench/results/transcripts/tier_b/whisper/
!git -C /content/Sauticivic commit -m "bench: add Tier B Whisper large-v3 transcripts (60 clips)"
!git -C /content/Sauticivic push https://{GH_TOKEN}@github.com/Spyder0000/Sauticivic.git main

[main 25f823c] bench: add Tier B Whisper large-v3 transcripts (60 clips)
 60 files changed, 1945 insertions(+)
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_001.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_002.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_003.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_004.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_005.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_006.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_007.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_008.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_009.json
 create mode 100644 bench/results/transcripts/tier_b/whisper/afrispeech_ng_010.json
 create mode 100644 bench/results/transcripts/tie